In [ ]:
# Ячейка 0 — что делать дальше (просто прочитайте вывод после Run)

from IPython.display import Markdown, display

display(
    Markdown(
        """
### GOT-OCR2.0 в Colab

1. **Runtime → Change runtime type → GPU** (настоятельно рекомендуется; на CPU очень медленно).
2. Выполните ячейки **1 → 4** по порядку. Первый запуск качает веса с Hugging Face.
3. Зависимости: `notes/requirements-ocr-notebook-colab.txt` (ячейка 1 ставит их через `pip`).

**Результаты:** `output/got_benchmark/` — тексты `hypotheses/got/*.txt`, `got_runs.jsonl`, `got_summaries.json` (ключ **`got_ocr2`**, те же поля метрик, что у MinerU/Paddle: CER, Final Score, WER в `_diagnostics`).

**Дополнительно:** развёрнутый ноутбук с пошаговыми ячейками и unit-тестами — [`notes/statisctics.ipynb`](statisctics.ipynb). **MinerU:** [`mineru_colab.ipynb`](mineru_colab.ipynb). **Paddle:** [`paddle_colab.ipynb`](paddle_colab.ipynb).

Ячейка **1** делает `git pull`; при конфликте с `output/*_benchmark` каталоги удаляются и pull повторяется. Ячейка **3** может скачать `got_image_benchmark.py` с Raw (**`OCR_ANALYZE_RAW_BASE`**), если в клоне нет скрипта.
"""
    )
)
print("Готово: выполняйте ячейку 1.")


In [ ]:
# Ячейка 1 — клон репозитория (Colab) + jiwer + зависимости GOT

from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую репозиторий…")
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочая папка:", Path.cwd().resolve())
    if (REPO_DIR / ".git").is_dir():

        def _pull() -> subprocess.CompletedProcess[str]:
            return subprocess.run(
                ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                capture_output=True,
                text=True,
            )

        pr = _pull()
        combined = (pr.stderr or "") + (pr.stdout or "")
        if pr.returncode != 0 and "would be overwritten by merge" in combined:
            for sub in ("mineru_benchmark", "paddle_benchmark", "got_benchmark"):
                d = REPO_DIR / "output" / sub
                if d.is_dir():
                    print("Удаляю (мешало git pull):", d)
                    shutil.rmtree(d, ignore_errors=True)
            pr = _pull()
            combined = (pr.stderr or "") + (pr.stdout or "")
        if pr.returncode == 0:
            print("git pull: OK")
        else:
            print("git pull: код", pr.returncode)
            print(combined[:1200] if combined else "(нет вывода)")
else:
    print("Не Colab — откройте ноутбук из корня репозитория. cwd:", Path.cwd().resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jiwer"])
req = Path("notes/requirements-ocr-notebook-colab.txt")
if req.is_file():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    print("OK: pip install -r", req)
else:
    print("WARN: нет", req.resolve())

print("OK. Следующая — ячейка 2 (пути и скрипт).")


In [ ]:
# Ячейка 2 — пути, список PNG, при необходимости скачать got_image_benchmark.py

from __future__ import annotations

import os
import urllib.error
import urllib.request
from pathlib import Path

REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "scripts").is_dir():
            return p
    cwd = Path.cwd().resolve()
    for start in [cwd, *cwd.parents]:
        if (start / "scripts" / "got_image_benchmark.py").is_file():
            return start
    return cwd


REPO_ROOT = find_repo_root()
INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
SCRIPT = REPO_ROOT / "scripts" / "got_image_benchmark.py"
OUT_DIR = REPO_ROOT / "output" / "got_benchmark"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT.resolve())
print("Скрипт:", SCRIPT.is_file(), SCRIPT)

if not SCRIPT.is_file():
    raw_base = os.environ.get(
        "OCR_ANALYZE_RAW_BASE",
        "https://raw.githubusercontent.com/developer-mixa/OCR-Analyze/main",
    ).rstrip("/")
    url = f"{raw_base}/scripts/got_image_benchmark.py"
    print("Скачиваю с Raw:\n ", url)
    SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, SCRIPT)
    except urllib.error.HTTPError as e:
        raise RuntimeError(
            f"HTTP {e.code} при скачивании. Запушьте scripts/got_image_benchmark.py или задайте OCR_ANALYZE_RAW_BASE."
        ) from e
    print("Скрипт скачан, байт:", SCRIPT.stat().st_size)

print("Входные PNG:", INPUT_DIR.resolve(), "— есть:", INPUT_DIR.is_dir())
print("Результаты:", OUT_DIR.resolve())
_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print("Найдено PNG:", len(_png))
for p in _png:
    stem = p.stem
    refs = [n for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md") if (INPUT_DIR / n).is_file()]
    print(" ", p.name, "| эталон:", ", ".join(refs) if refs else "нет")
if not _png:
    print("Добавьте PNG в input/data/1.")
print("Следующая — ячейка 3 (model_id / ocr_type), затем 4.")


In [ ]:
# Ячейка 3 — параметры GOT-OCR2 (как в statisctics.ipynb)

GOT_MODEL_ID = "ucaslcl/GOT-OCR2_0"
# Режимы из README модели: "ocr" | "format"
GOT_OCR_TYPE = "ocr"

print("GOT_MODEL_ID =", GOT_MODEL_ID)
print("GOT_OCR_TYPE =", GOT_OCR_TYPE)
print("После смены снова выполните ячейку 4.")


In [ ]:
# Ячейка 4 — прогон GOT по всем PNG (долго: загрузка модели + inference)

import json
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "SCRIPT" not in globals():
    raise RuntimeError("Сначала ячейка 2.")
if not SCRIPT.is_file():
    raise RuntimeError("Нет scripts/got_image_benchmark.py — см. ячейку 2.")

mid = globals().get("GOT_MODEL_ID", "ucaslcl/GOT-OCR2_0")
ocr_t = globals().get("GOT_OCR_TYPE", "ocr")

argv = [
    sys.executable,
    str(SCRIPT),
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUT_DIR),
    "--model-id",
    str(mid),
    "--ocr-type",
    str(ocr_t),
]

print("Запуск GOT, model =", mid, ", ocr_type =", ocr_t)
print("Пишем в", OUT_DIR)
subprocess.check_call(argv, cwd=str(REPO_ROOT))

hyp_dir = OUT_DIR / "hypotheses" / "got"
print("\nГипотезы (.txt):")
for p in sorted(hyp_dir.glob("*.txt")):
    print(" ", p.name, p.stat().st_size, "байт")
if not list(hyp_dir.glob("*.txt")):
    print("  нет .txt — см. got_runs.jsonl (поле error)")

for name in ("got_hypotheses_raw.json", "got_hypotheses_concat.txt"):
    fp = OUT_DIR / name
    if fp.is_file():
        print("Доп. файл:", fp.name, fp.stat().st_size, "байт")

summ = OUT_DIR / "got_summaries.json"
if summ.is_file():
    print("\n---", summ.name, "---")
    txt = summ.read_text(encoding="utf-8")
    print(txt)
    try:
        outs = json.loads(txt).get("got_ocr2", {}).get("_outputs")
        if outs:
            print("\n_outputs:", json.dumps(outs, ensure_ascii=False, indent=2))
    except json.JSONDecodeError:
        pass
